# Working with CLAMPS data — instrument overview example

This notebook walks through how to **load plot-ready CLAMPS inputs** and build the gallery **4-panel instrument figure** for one case (`ci_c1`, NWCRIL2020 CLAMPS1, 2020-07-30). The same observational day appears in the gallery as **`gravity_waves_c1`**.

The steps mirror the production pipeline in `code/case_gallery/plot_instrument.py`, but each stage is shown here so you can adapt the pattern to other cases or panels.

**Data layout** (after download):

| Subfolder | Product | Role in this figure |
|-----------|---------|---------------------|
| `tropoe/` | TROPoe | θ_v and q profiles (panels A & B) |
| `dlfp/` | DLFP | Vertical-velocity variance σ²_w (panel D) |
| `dlvad/` | DLVAD | SNR ceiling overlay on wind panel |
| `windoe/` | WINDoe | Horizontal wind (panel C) — precomputed, not re-run here |
| `pbl_fuzzy/` | fuzzy PBLH | Black PBL height line on all panels |

Cloud-base markers use TROPoe **`lwp > 5` g m⁻²**. WINDoe and fuzzy PBL files are bundled for this example; a full pipeline would run those retrievals separately.

## Setup

In [ ]:
import os
import subprocess
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

from IPython.display import Image, display, Markdown

ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent):
    if (candidate / "code" / "case_gallery").is_dir():
        ROOT = candidate
        break

CODE = ROOT / "code"
sys.path.insert(0, str(CODE))
os.environ["MPLCONFIGDIR"] = str(ROOT / ".matplotlib")
(ROOT / ".matplotlib").mkdir(exist_ok=True)

# Download example data (GitHub Release) if needed, then symlink into pipeline layout.
subprocess.run(["bash", str(ROOT / "scripts" / "link_data.sh")], check=True)
print(f"Project root: {ROOT}")

## 1. Case metadata and input files

Case dates and campaigns live in `code/case_gallery/cases.yaml`. The helper resolves **staged paths** under `output/case_gallery/` (symlinked to `data/ci_c1/` here).

In [ ]:
from case_gallery.case_lib import find_case_files, find_pbl_file, find_windoe_file, get_case

CASE_ID = "ci_c1"
case = get_case(CASE_ID)
files = find_case_files(case)
pbl_path = find_pbl_file(case)
windoe_path = find_windoe_file(case)

display(Markdown(
    f"**{case.project}** {case.platform} · **{case.case_date.isoformat()}** · "
    f"category `{case.category}`"
))

for label, path in [
    ("TROPoe", files.tropoe),
    ("DLFP", files.dlfp),
    ("DLVAD", files.dlvad),
    ("DL PPI", files.dlppi),
    ("WINDoe", windoe_path),
    ("PBL fuzzy", pbl_path),
]:
    print(f"{label:10} {path}")

## 2. Horizontal wind (WINDoe)

Gallery cases use **precomputed WINDoe** NetCDFs. `load_case_winds` reads the file for the case date; if WINDoe were missing, the helper could fall back to DLVAD.

In [ ]:
from case_gallery.winds import load_case_winds
from plot_awaken_instrument_template import wind_panel_title

wind, wind_source = load_case_winds(case, prefer="auto")
wind_title = wind_panel_title(wind_source)

print(f"Wind source: {wind_source}")
print(f"Panel title: {wind_title}")
print(f"Times: {wind.hour.size}  ·  heights ≤3 km: {wind.height_km[wind.height_km <= 3].size}")

## 3. Thermodynamic cross-section (TROPoe) and fuzzy PBL height

TROPoe provides θ_v and q on a height–time grid. DLFP variance (next section) shares the fuzzy PBL time axis for masking gaps.

In [ ]:
from awaken_la_diagnostics import CROSS_SECTION_MAX_KM, build_profiler_cross_section, load_fuzzy_pblh

prof = build_profiler_cross_section(
    files.tropoe,
    files.dlfp,
    max_km=CROSS_SECTION_MAX_KM,
    apply_tropoe_qc=False,
)
pbl = load_fuzzy_pblh(pbl_path)

print(f"TROPoe grid: {prof.tropoe_hour.size} times × {prof.tropoe_height_km.size} heights (to {CROSS_SECTION_MAX_KM:g} km)")
print(f"PBLH series: {pbl.hour_decimal.size} points")

## 4. DLVAD SNR ceiling and cloud-base markers

The wind panel masks weak DL returns using the **DLVAD SNR ceiling**. Cloud-base dots come from TROPoe `cbh`, gated on liquid water path **`lwp > 5` g m⁻²** (see `load_cloud_base` in `plot_awaken_instrument_template.py`).

In [ ]:
from awaken_windoe import load_dlvad_snr
from plot_awaken_instrument_template import (
    DLVAD_SNR_THRESHOLD,
    dlvad_snr_ceiling,
    load_cloud_base,
    smooth_snr_ceiling,
)

dlvad_snr = load_dlvad_snr(files.dlvad)
snr_h, snr_z = dlvad_snr_ceiling(dlvad_snr, threshold=DLVAD_SNR_THRESHOLD)
snr_z = smooth_snr_ceiling(snr_h, snr_z)

cb_h, cb_z = load_cloud_base(files.tropoe)
print(f"SNR ceiling: {snr_h.size} points")
print(f"Cloud-base markers (lwp>5 g m⁻²): {cb_h.size}")

## 5. Color limits and 24-hour UTC axis

Per-case color limits are tabulated in `plot_limits.py`. The x-axis spans 00–24 UTC on the case date.

In [ ]:
from case_gallery.plot_limits import limits_for_case, q_ticks, theta_v_ticks, wspd_ticks
from plot_awaken_instrument_template import DEFAULT_SIGMA_WSPD_MAX, la

limits = limits_for_case(CASE_ID)
case_date = case.case_date
start = datetime(case_date.year, case_date.month, case_date.day, tzinfo=timezone.utc)
period = la.PeriodAxis(start=start, end=start + timedelta(hours=24))

suptitle = (
    f"{case.project} CLAMPS{case.platform[-1]}\n"
    f"{case_date.isoformat()} · CLAMPS observations · PBLH (fuzzy logic)"
)

out_dir = case.figure_dir
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "instrument_template_4panel.png"

print(f"θ_v limits: {limits.theta_v}")
print(f"q limits (g kg⁻¹): {limits.q_gkg}")
print(f"Wind speed limits: {limits.wspd}")
print(f"Output: {out_path}")

## 6. Draw the four panels

`plot_four_panel_template` handles matplotlib layout only — all science inputs were prepared above.

In [ ]:
from plot_awaken_instrument_template import plot_four_panel_template

plot_four_panel_template(
    prof,
    wind,
    snr_h,
    snr_z,
    pbl,
    case_date,
    period,
    out_path,
    sigma_wspd_max=DEFAULT_SIGMA_WSPD_MAX,
    suptitle=suptitle,
    theta_v_lim=limits.theta_v,
    q_lim=limits.q_gkg,
    wspd_lim=limits.wspd,
    theta_v_cbar_ticks=theta_v_ticks(*limits.theta_v),
    q_cbar_ticks=q_ticks(*limits.q_gkg),
    wspd_cbar_ticks=wspd_ticks(*limits.wspd),
    cloud_base_hour=cb_h,
    cloud_base_km=cb_z,
    wind_panel_title=wind_title,
    tropoe_path=files.tropoe,
    dlfp_path=files.dlfp,
)
print(f"Saved {out_path}")

In [ ]:
display(Image(filename=str(out_path)))

## Next steps

- Change `CASE_ID` and re-run (after staging another case under `data/`).
- Batch all cases: `python code/case_gallery/plot_instrument.py <case_id> --force` wraps the same steps.
- Full HPC pipeline (THREDDS download, WINDoe, fuzzy PBL): see the processing workspace `clamps_viz_process` on the server.